# CineOS — ComfyUI + FLUX Cloud GPU Worker

This notebook runs a **ComfyUI + FLUX/SDXL** GPU worker on Google Colab.
It exposes a REST API that the CineOS Cloud Worker Bridge dispatches jobs to.

**Requirements:**
- Free [ngrok](https://ngrok.com) account for the public tunnel
- Runtime: **GPU (T4 or better)**

**Steps:**
1. Set your `NGROK_AUTH_TOKEN` below
2. Run **all cells** (Runtime → Run all)
3. Copy the printed `COLAB_COMFYUI_ENDPOINT` URL into your server's `.env`

Auto-shutdown after 15 minutes of inactivity saves your GPU quota.

In [ ]:
#@title 1. Configuration
#@markdown Set your ngrok auth token (free at https://ngrok.com)
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
#@markdown API key — must match `COLAB_API_KEY` in your server `.env`
CINEOS_API_KEY = "" #@param {type:"string"}
#@markdown Auto-shutdown after this many minutes of no requests
INACTIVITY_TIMEOUT_MINUTES = 15 #@param {type:"integer"}

COMFYUI_PORT = 8188
API_PORT = 8199

assert NGROK_AUTH_TOKEN, "Set your NGROK_AUTH_TOKEN above"
print(f"Config: API port={API_PORT}, shutdown after {INACTIVITY_TIMEOUT_MINUTES}min")

In [ ]:
#@title 2. Install Dependencies
import subprocess, sys, os

def run(cmd, desc=""):
    if desc: print(f"Installing {desc}...")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print(f"  Warning: {r.stderr[:200]}")
    return r.returncode == 0

run("apt-get update -qq", "system packages")
run("apt-get install -y -qq aria2 libgl1-mesa-glx libglib2.0-0", "system libs")
run(f"{sys.executable} -m pip install -q flask flask-cors pyngrok requests", "Flask + ngrok")

# RealESRGAN
if not os.path.exists("/content/Real-ESRGAN"):
    run("git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN", "RealESRGAN")
    run(f"{sys.executable} -m pip install -q basicsr facexlib gfpgan", "RealESRGAN deps")
    run(f"{sys.executable} -m pip install -q -e /content/Real-ESRGAN", "RealESRGAN install")

# ComfyUI
if not os.path.exists("/content/ComfyUI"):
    run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI", "ComfyUI")
    run(f"{sys.executable} -m pip install -q -r /content/ComfyUI/requirements.txt", "ComfyUI deps")

# ComfyUI Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    run("git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager", "ComfyUI Manager")

print("\nCore dependencies installed")

In [ ]:
#@title 3. Install Custom Nodes (IP-Adapter, FaceDetailer, ControlNet)
import os, subprocess, sys

CUSTOM_NODES_DIR = "/content/ComfyUI/custom_nodes"
nodes = {
    "ComfyUI-IPAdapter-Plus": "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "ComfyUI-FaceDetailer": "https://github.com/dsdish/comfyui-FaceDetailer.git",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
}
for name, url in nodes.items():
    target = f"{CUSTOM_NODES_DIR}/{name}"
    if not os.path.exists(target):
        subprocess.run(f"git clone --depth 1 {url} {target}", shell=True, capture_output=True)
    req = f"{target}/requirements.txt"
    if os.path.exists(req):
        subprocess.run(f"{sys.executable} -m pip install -q -r {req}", shell=True, capture_output=True)

print("Custom nodes installed")

In [ ]:
#@title 4. Download Models (SDXL, CLIP, ControlNet, IP-Adapter, RealESRGAN, LoRA)
import os, subprocess

MODELS = "/content/ComfyUI/models"
downloads = [
    ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors", f"{MODELS}/checkpoints/sd_xl_base_1.0.safetensors", "SDXL Base 1.0"),
    ("https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors", f"{MODELS}/vae/sdxl_vae.safetensors", "SDXL VAE"),
    ("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors", f"{MODELS}/clip/clip_l.safetensors", "CLIP-L"),
    ("https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors", f"{MODELS}/clip/t5xxl_fp16.safetensors", "T5-XXL"),
    ("https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth", f"{MODELS}/controlnet/control_v11p_sd15_canny.pth", "ControlNet Canny"),
    ("https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_openpose.pth", f"{MODELS}/controlnet/control_v11p_sd15_openpose.pth", "ControlNet OpenPose"),
    ("https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sdxl_vit-h.safetensors", f"{MODELS}/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors", "IP-Adapter Plus"),
    ("https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors", f"{MODELS}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors", "CLIP Vision"),
    ("https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4plus.pth", f"{MODELS}/upscale_models/RealESRGAN_x4plus.pth", "RealESRGAN x4"),
    ("https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4plus_anime_6B.pth", f"{MODELS}/upscale_models/RealESRGAN_x4plus_anime_6B.pth", "RealESRGAN anime"),
    ("https://huggingface.co/InstantX/FLUX.1-dev-LoRA-Add-details/resolve/main/add_details.safetensors", f"{MODELS}/loras/add_details.safetensors", "FLUX detail LoRA"),
]
for url, dest, desc in downloads:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {desc}...")
        subprocess.run(f'aria2c -x 16 -s 16 -k 1M -d "{os.path.dirname(dest)}" -o "{os.path.basename(dest)}" "{url}"', shell=True, capture_output=True)
    else:
        print(f"  {desc} cached")

print("\nAll models downloaded")

In [ ]:
#@title 5. Start ComfyUI + REST API + ngrok
import threading, uuid, hashlib, base64, json, signal, os, sys, time, subprocess
from pathlib import Path
from flask import Flask, request, jsonify
from flask_cors import CORS
import requests as http_requests
from pyngrok import ngrok

COMFYUI_DIR = "/content/ComfyUI"
_last_activity = time.monotonic()
_shutting_down = False
_jobs = {}
_jobs_count = 0
_start_time = time.monotonic()

# Start ComfyUI
comfyui_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(COMFYUI_PORT), "--dont-print-server"],
    cwd=COMFYUI_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(f"ComfyUI starting (PID: {comfyui_proc.pid})...")
for i in range(60):
    try:
        r = http_requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/system_stats", timeout=2)
        if r.status_code == 200:
            print(f"ComfyUI ready in {i+1}s")
            break
    except Exception: pass
    time.sleep(1)

# Inactivity timer
def _check_inactivity():
    global _shutting_down
    while not _shutting_down:
        if time.monotonic() - _last_activity > INACTIVITY_TIMEOUT_MINUTES * 60:
            print(f"\n[INACTIVITY] Shutting down after {INACTIVITY_TIMEOUT_MINUTES}min")
            _shutting_down = True
            os.kill(os.getpid(), signal.SIGTERM)
            break
        time.sleep(30)
threading.Thread(target=_check_inactivity, daemon=True).start()

# Flask API
api = Flask(__name__)
CORS(api)

@api.route("/health")
def health():
    try:
        r = http_requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/system_stats", timeout=5)
        ok = r.status_code == 200
    except Exception: ok = False
    return jsonify({"status": "healthy" if ok else "degraded", "uptime": round(time.monotonic()-_start_time,1)})

@api.route("/warmup", methods=["POST"])
def warmup():
    global _last_activity; _last_activity = time.monotonic()
    return jsonify({"status": "ok"})

@api.route("/job", methods=["POST"])
def submit_job():
    global _last_activity, _jobs_count; _last_activity = time.monotonic(); _jobs_count += 1
    if CINEOS_API_KEY and request.headers.get("X-Api-Key") != CINEOS_API_KEY:
        return jsonify({"error": "Unauthorized"}), 401
    data = request.get_json()
    if not data: return jsonify({"error": "No JSON body"}), 400
    task_id = data.get("task_id", str(uuid.uuid4()))
    job_type = data.get("job_type", "image_generation")
    payload = data.get("payload", {})
    if job_type == "image_generation":
        prompt = payload.get("prompt", "")
        neg = payload.get("negative_prompt", "")
        w, h, steps, cfg = payload.get("width",1024), payload.get("height",1024), payload.get("steps",30), payload.get("cfg_scale",7.0)
        seed = payload.get("seed") or int.from_bytes(os.urandom(4), "big")
        wf = {"3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":steps,"cfg":cfg,"sampler_name":"euler_ancestral","scheduler":"normal","denoise":1.0,"model":["4",0],"positive":["6",0],"negative":["7",0],"latent_image":["5",0]}},"4":{"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":payload.get("model","sd_xl_base_1.0.safetensors")}},"5":{"class_type":"EmptyLatentImage","inputs":{"width":w,"height":h,"batch_size":1}},"6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["4",1]}},"7":{"class_type":"CLIPTextEncode","inputs":{"text":neg,"clip":["4",1]}},"8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["4",2]}},"9":{"class_type":"SaveImage","inputs":{"filename_prefix":f"cineos_{task_id[:8]}","images":["8",0]}}}
        _jobs[task_id] = {"status": "processing"}
        threading.Thread(target=_run_comfyui, args=(task_id, wf), daemon=True).start()
    else:
        return jsonify({"error": f"Unknown job_type: {job_type}"}), 400
    return jsonify({"task_id": task_id, "status": "processing"})

@api.route("/status/<task_id>")
def task_status(task_id):
    return jsonify(_jobs.get(task_id, {"error": "not found"}))

def _run_comfyui(task_id, wf):
    try:
        r = http_requests.post(f"http://127.0.0.1:{COMFYUI_PORT}/prompt", json={"prompt": wf}, timeout=30)
        r.raise_for_status(); pid = r.json().get("prompt_id")
        if not pid: _jobs[task_id] = {"status":"failed","error":"no prompt_id"}; return
        for _ in range(300):
            time.sleep(1)
            h = http_requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/history/{pid}", timeout=10)
            if h.status_code != 200 or pid not in h.json(): continue
            for nid, no in h.json()[pid].get("outputs",{}).items():
                for img in no.get("images",[]):
                    ir = http_requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/view", params={"filename":img["filename"],"subfolder":img.get("subfolder",""),"type":img.get("type","output")}, timeout=30)
                    ir.raise_for_status(); ib = ir.content
                    os.makedirs("/content/output", exist_ok=True)
                    with open(f"/content/output/{task_id}.png","wb") as f: f.write(ib)
                    _jobs[task_id] = {"status":"completed","result":{"image_base64":base64.b64encode(ib).decode(),"checksum":hashlib.sha256(ib).hexdigest(),"source":"comfyui_colab"}}
                    return
        _jobs[task_id] = {"status":"failed","error":"timeout 300s"}
    except Exception as e: _jobs[task_id] = {"status":"failed","error":str(e)}

threading.Thread(target=lambda: api.run(host="0.0.0.0", port=API_PORT, debug=False, use_reloader=False), daemon=True).start()

# ngrok tunnel
if NGROK_AUTH_TOKEN: ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(API_PORT, "http")
public_url = tunnel.public_url

print(f"\n{'='*60}")
print(f"CineOS Colab Worker is LIVE!")
print(f"{'='*60}")
print(f"Public URL: {public_url}")
print(f"Health:     {public_url}/health")
print(f"Submit:     {public_url}/job")
print(f"{'='*60}")
print(f"\nAdd to your server .env:")
print(f"  COLAB_COMFYUI_ENDPOINT={public_url}")
print(f"\nAuto-shutdown in {INACTIVITY_TIMEOUT_MINUTES}min of inactivity")

In [ ]:
#@title 6. Keep Alive (run to keep worker running)
import signal, os
def _sig(s,f): print("\nShutting down..."); ngrok.kill(); comfyui_proc.terminate(); os._exit(0)
signal.signal(signal.SIGTERM, _sig); signal.signal(signal.SIGINT, _sig)
print(f"Worker running at {public_url}")
print("Press Ctrl+C in Colab to stop")
while not _shutting_down:
    try: time.sleep(10)
    except KeyboardInterrupt: break
ngrok.kill(); comfyui_proc.terminate(); print("Worker stopped")